## Configuração do Escopo de Banco de Dados

In [0]:
%sql
-- Camada Silver: Processamento e Normalização cadastro
-- **Descrição:** Este script lê os dados semiestruturados da camada Bronze, realiza o parsing dos campos JSON e popula o schema Silver de forma desnormalizada.
-- **Origem:** Bronze
-- **Destino:** Silver
-- **Observações:** 
-- Criação Rodrigo Escarmeloto Franco

-- Definição do catálogo de trabalho
USE CATALOG portfolio_mercado_livre;

-- Criação automática do Schema Silver se não existir
CREATE SCHEMA IF NOT EXISTS silver;

## Criação e Carga da Trusted Cliente

In [0]:
%sql
--  1. Tabela trusted: Clientes (trusted_clientes)
-- tabela contém os campos de data de inserção e atualização da tabela junto com a flag de ultima versão para garantir a integridade do dado

-- Criação da tabela Delta estruturada na Silver
CREATE OR REPLACE TABLE silver.trusted_clientes (
    sk_cliente STRING
    ,id_cliente STRING
    ,nickname_cliente STRING
    ,tipo_documento STRING
    ,estado_entrega STRING
    ,cidade_entrega STRING
    ,dh_insercao_silver TIMESTAMP
    ,dh_atualizacao_silver TIMESTAMP
    ,fl_ultima_versao SMALLINT
) USING DELTA;

-- Carga incremental/Sobrescrita da trusted Clientes
-- Fazendo o parsing dinâmico da coluna string JSON vinda da Bronze
INSERT OVERWRITE silver.trusted_clientes
SELECT DISTINCT
    hash(get_json_object(buyer, '$.id') ) as sk_cliente
    ,get_json_object(buyer, '$.id') AS id_cliente
    ,get_json_object(buyer, '$.nickname') AS nickname_cliente
    ,get_json_object(buyer, '$.billing_info.doc_type') AS tipo_documento
    ,get_json_object(buyer, '$.shipping_address.state') AS estado_entrega
    ,get_json_object(buyer, '$.shipping_address.city') AS cidade_entrega
    ,current_timestamp() AS dh_insercao_silver
    ,cast("2100-01-01T00:00:00.000+00:00" as timestamp) as dh_atualizacao_silver
    ,1 fl_ultima_versao
FROM bronze.mercado_livre_vendas
WHERE get_json_object(buyer, '$.id') IS NOT NULL;

-- Prévia dos dados normalizados de clientes e regiões
SELECT * FROM silver.trusted_clientes LIMIT 5;

## Criação e Carga da Trusted Produto

In [0]:
%sql
-- 2. Tabela trusted: Produtos (trusted_produtos)
-- tabela contém os campos de data de inserção e atualização da tabela junto com a flag de ultima versão para garantir a integridade do dado

-- DROP TABLE IF EXISTS portfolio_mercado_livre.silver.trusted_produtos;

CREATE OR REPLACE TABLE silver.trusted_produtos (
    sk_produto STRING
    ,id_produto string
    ,preco_unitario DOUBLE
    ,quantidade_itens INT
    ,origem_dados STRING
    ,dh_ingestao_etl TIMESTAMP
    ,dh_insercao_silver TIMESTAMP
    ,dh_atualizacao_silver TIMESTAMP
    ,fl_ultima_versao SMALLINT
) USING DELTA;

-- Agrupando todos os produtos únicos da plataforma
INSERT OVERWRITE silver.trusted_produtos
with venda_cliente as (
  select  
    replace(get_json_object(order_items, '$[*].item_id'), '"',"") AS id_produto
    ,get_json_object(order_items, '$[*].quantity') AS quantidade_itens
    ,get_json_object(order_items, '$[*].unit_price') AS preco_unitario
from bronze.mercado_livre_vendas
), produto as (
    SELECT 
      id_item_referencia as id_produto
      ,dh_ingestao_etl
      ,origem_dados 
    FROM bronze.mercado_livre_produtos
)
  select distinct
    hash(a.id_produto) as sk_produto
    ,a.id_produto
    ,b.preco_unitario
    ,b.quantidade_itens
    ,a.origem_dados
    ,a.dh_ingestao_etl
    ,current_timestamp() as dh_insercao_silver
    ,cast("2100-01-01T00:00:00.000+00:00" as timestamp) as dh_atualizacao_silver
    ,1 fl_ultima_versao
  from produto a
  join venda_cliente b 
  on a.id_produto = b.id_produto
WHERE a.id_produto IS NOT NULL;

SELECT * FROM silver.trusted_produtos LIMIT 5;

## Criação e Carga da Trusted Status Venda


In [0]:
%sql
-- 3. Tabela trusted: status venda (trusted_status_venda)
-- tabela contém os campos de data de inserção e atualização da tabela junto com a flag de ultima versão para garantir a integridade do dado

CREATE OR REPLACE TABLE silver.trusted_status_venda (
    sk_status_venda STRING
    ,status_venda STRING
    ,fl_pagamento SMALLINT
    ,dh_insercao_silver TIMESTAMP
    ,dh_atualizacao_silver TIMESTAMP
    ,fl_ultima_versao SMALLINT
) USING DELTA;

-- Agrupando todos os produtos únicos da plataforma
WITH status AS (
    SELECT distinct 
        status AS status_venda
    FROM bronze.mercado_livre_vendas
)
INSERT OVERWRITE silver.trusted_status_venda
SELECT DISTINCT
    hash(status_venda) sk_status_venda 
    ,status_venda
    ,case when status_venda = "paid" then 1 else 0 end as fl_pagamento
    ,current_timestamp() AS dh_insercao_silver
    ,cast("2100-01-01T00:00:00.000+00:00" as timestamp)
    , 1 fl_ultima_versao
FROM status;

-- Prévia da tabela evento para auditoria de métricas
SELECT * FROM silver.trusted_status_venda LIMIT 5;

## Criação e Carga da Trusted calendario


In [0]:
%sql
-- 4. Tabela evento: Vendas (evento_venda)
-- tabela contém os campos de data de inserção e atualização da tabela junto com a flag de ultima versão para garantir a integridade do dado

CREATE OR REPLACE TABLE silver.trusted_calendario (
    sk_calendario INT
    ,id_data DATE
    ,data_venda TIMESTAMP
    ,ano STRING
    ,mes STRING
    ,dia STRING
    ,dia_semana STRING
    ,trimestre STRING
    ,semestre STRING
) USING DELTA;


WITH trusted_calendario AS (
    SELECT distinct
        CAST(date_created AS TIMESTAMP) AS data_venda
    FROM bronze.mercado_livre_vendas
)
INSERT OVERWRITE silver.trusted_calendario
SELECT DISTINCT
    cast(DATE_FORMAT(data_venda, 'yyyyMMdd') as int) AS sk_calendario
    ,cast(DATE_FORMAT(data_venda, 'yyyy-MM-dd') as date) AS id_data
    ,data_venda
    ,year(data_venda) as ano
    ,month(data_venda) as mes 
    ,day(data_venda) as dia
    ,date_format(data_venda,'E') as dia_semana
    ,case when month(data_venda) >= 1 and month(data_venda) <= 3 then 'Q1'
          when month(data_venda) >= 4 and month(data_venda) <= 6 then 'Q2'
          when month(data_venda) >= 7 and month(data_venda) <= 9 then 'Q3'
          when month(data_venda) >= 10 and month(data_venda) <= 12 then 'Q4'
          else 'N/A'end as trimestre
    ,case when month(data_venda) >= 1 and month(data_venda) <= 6 then '1S'
          when month(data_venda) >= 7 and month(data_venda) <= 12 then '2S'
          else 'N/A'end as semestre
FROM trusted_calendario;

-- Prévia da tabela evento para auditoria de métricas
SELECT * FROM silver.trusted_calendario LIMIT 5;

## Criação e Carga da Trusted eventos_vendas (Com Explode de Itens)

In [0]:
%sql
-- 5. Tabela evento: Vendas (evento_venda)
CREATE OR REPLACE TABLE silver.evento_vendas (
    id_venda STRING
    ,sk_calendario INT
    ,id_produto STRING
    ,id_cliente STRING
    ,sk_cliente STRING
    ,sk_produto STRING
    ,sk_status_venda STRING
    ,quantidade_itens INT
    ,valor_unitario DOUBLE
    ,valor_total_venda DOUBLE
    ,dh_insercao_silver TIMESTAMP
) USING DELTA;

-- Carga da evento com extração do Array de Itens Comprados
WITH vendas_parsed AS (
    SELECT 
        id AS id_venda
        ,cast(DATE_FORMAT(date_created, 'yyyyMMdd') as int) AS sk_calendario
        ,get_json_object(buyer, '$.id') AS id_cliente
        ,hash(get_json_object(buyer, '$.id')) sk_cliente
        ,status AS status_venda
        ,CAST(total_amount AS DOUBLE) AS valor_total_venda
        -- Mapeia a estrutura do array JSON interno de itens da venda para que o SQL entenda
        ,from_json(order_items, 'ARRAY<STRUCT<item_id: STRING, quantity: INT, unit_price: DOUBLE>>') AS itens_array
    FROM bronze.mercado_livre_vendas
)
INSERT OVERWRITE silver.evento_vendas
SELECT 
    id_venda
    ,sk_calendario
    ,item_desaninhado.item_id AS id_produto
    ,id_cliente
    ,sk_cliente
    ,hash(status_venda) sk_status_venda
    ,hash(item_desaninhado.item_id) AS sk_produto
    ,item_desaninhado.quantity AS quantidade_itens
    ,item_desaninhado.unit_price AS valor_unitario
    ,valor_total_venda
    ,current_timestamp() AS dh_insercao_silver
-- O EXPLODE quebra o Array de produtos, gerando linhas individuais se o cliente comprou mais de 1 produto diferente
FROM vendas_parsed
LATERAL VIEW explode(itens_array) AS item_desaninhado;

-- Prévia da tabela evento para auditoria de métricas
SELECT * FROM silver.evento_vendas LIMIT 5;